# NB05 — Integration: Observed + SoilGrids Fallback

**Objective**: Merge the WoSIS observed dataset (NB03) with the SoilGrids 250m raster extraction
(NB04) into a unified surface soil dataset. Observed values always take priority; SoilGrids
values fill in only where observations are absent.

**Integration rules** (applied per station × property):
1. **Observed available** → use observed value, flag = `observed`
2. **Observed NaN, SoilGrids available** → use SoilGrids value, flag = `soilgrids`
3. **Both NaN** → NaN, flag = `missing`

**Properties covered by SoilGrids fallback** (11 properties):

| WoSIS property | SoilGrids column   | Notes |
|----------------|--------------------|-------|
| BD             | sg_BD              | g/cm³ |
| CEC            | sg_CEC             | cmol(c)/kg |
| CF             | sg_CF              | % |
| clay           | sg_clay            | g/kg |
| N              | sg_N               | g/kg |
| occ            | sg_occ_carbon      | g/kg (soc layer, same unit as WoSIS occ) |
| pH             | sg_pH              | pH |
| sand           | sg_sand            | g/kg |
| silt           | sg_silt            | g/kg |
| WR_volumetric  | sg_WR_volumetric   | cm³/cm³ (0.01 bar) |
| WR_gravimetric | sg_WR_gravimetric  | cm³/cm³ (15 bar) |

**Properties observed-only** (no SoilGrids equivalent, no fallback):
Al, Ca, CaCO3, Cu, EC, Fe, K, Mg, Mn, Na, nematode, P, TC

**Additional SoilGrids-only columns** (kept in output, no WoSIS counterpart):
- `sg_occ_ocd` — organic carbon density from SoilGrids ocd layer (g/dm³)
- `sg_WR_volumetric2` — water retention at 0.33 bar (cm³/cm³)

**Inputs**:
- `data/intermediate/wosis_surface_qc.parquet` (NB03 — observed values + QC_FLAG)
- `data/intermediate/soilgrids_surface_all.parquet` (NB04 — SoilGrids values)

**Outputs**:
- `data/intermediate/unified_surface_all.parquet` — merged, all properties, source-flagged
- `data/intermediate/unified_source_flags.parquet` — long format: station_id × property → source
- `logs/nb05_integration.log`

**USABLE stations only**: QC_FLAG ∈ {VALID, NEAR_BORDER, OVERSEAS_TERRITORY}

In [1]:
import logging
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

# ── Paths ──────────────────────────────────────────────────────────────────
BASE      = Path('/home/agbelgaid/Documents/WORKSPACE/DataCollection/SoilHive')
INTER_DIR = BASE / 'data' / 'intermediate'
LOG_DIR   = BASE / 'logs'

# ── Logging ────────────────────────────────────────────────────────────────
log_path = LOG_DIR / 'nb05_integration.log'
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.FileHandler(log_path, mode='w'),
        logging.StreamHandler()
    ]
)
log = logging.getLogger('nb05')
log.info(f'NB05 started at {datetime.now().isoformat()}')

# ── Usable QC flags (must match NB03) ─────────────────────────────────────
USABLE_FLAGS = {'VALID', 'NEAR_BORDER', 'OVERSEAS_TERRITORY'}

# ── SoilGrids → WoSIS property mapping ────────────────────────────────────
# Format: wosis_property → (sg_column, scale_factor)
# scale_factor converts SoilGrids units → WoSIS units before the fallback merge.
#
# WR unit mismatch (confirmed by NB06 validation):
#   SoilGrids wv* layers are in cm³/cm³ (after ÷1000 divisor), range ~0.1–0.6
#   WoSIS WR_volumetric and WR_gravimetric are in % (volumetric water content × 100)
#   Observed median ≈ 30%; SoilGrids median ≈ 0.37  →  scale_factor = 100
#
# All other properties: units already match, scale_factor = 1.0
SG_FALLBACK_MAP = {
    'BD':             ('sg_BD',              1.0),
    'CEC':            ('sg_CEC',             1.0),
    'CF':             ('sg_CF',              1.0),
    'clay':           ('sg_clay',            1.0),
    'N':              ('sg_N',               1.0),
    'occ':            ('sg_occ_carbon',      1.0),   # WoSIS occ [g/kg] ↔ SoilGrids soc [g/kg]
    'pH':             ('sg_pH',              1.0),
    'sand':           ('sg_sand',            1.0),
    'silt':           ('sg_silt',            1.0),
    'WR_volumetric':  ('sg_WR_volumetric',  100.0),  # cm³/cm³ → %  (×100)
    'WR_gravimetric': ('sg_WR_gravimetric', 100.0),  # cm³/cm³ → %  (×100)
}

# WoSIS properties with NO SoilGrids equivalent — observed only
OBS_ONLY_PROPS = ['Al', 'Ca', 'CaCO3', 'Cu', 'EC', 'Fe', 'K', 'Mg', 'Mn', 'Na', 'nematode', 'P', 'TC']

# SoilGrids-only columns kept in output (no WoSIS counterpart, no fallback role)
SG_ONLY_COLS = {
    'sg_occ':          'sg_occ_ocd',       # organic carbon density (g/dm³) — renamed for clarity
    'sg_WR_volumetric2': 'sg_WR_volumetric2',  # water retention at 0.33 bar (cm³/cm³)
}

print('SoilGrids fallback properties :', len(SG_FALLBACK_MAP))
print('Observed-only properties       :', len(OBS_ONLY_PROPS))
for prop, (sg_col, scale) in SG_FALLBACK_MAP.items():
    scale_str = f'×{scale}' if scale != 1.0 else ''
    print(f'  {prop:20s} ← {sg_col}  {scale_str}')

2026-03-19 23:18:43,193 [INFO] NB05 started at 2026-03-19T23:18:43.193645


SoilGrids fallback properties : 11
Observed-only properties       : 13
  BD                   ← sg_BD  
  CEC                  ← sg_CEC  
  CF                   ← sg_CF  
  clay                 ← sg_clay  
  N                    ← sg_N  
  occ                  ← sg_occ_carbon  
  pH                   ← sg_pH  
  sand                 ← sg_sand  
  silt                 ← sg_silt  
  WR_volumetric        ← sg_WR_volumetric  ×100.0
  WR_gravimetric       ← sg_WR_gravimetric  ×100.0


In [2]:
# ── Load inputs ────────────────────────────────────────────────────────────

# Observed surface values + QC flags (from NB03)
qc = pd.read_parquet(INTER_DIR / 'wosis_surface_qc.parquet')
obs = qc[qc['QC_FLAG'].isin(USABLE_FLAGS)].copy()
log.info(f'Observed stations loaded: {len(obs):,} (after QC filter)')

# SoilGrids raster extraction (from NB04)
sg = pd.read_parquet(INTER_DIR / 'soilgrids_surface_all.parquet')
log.info(f'SoilGrids stations loaded: {len(sg):,}')

# Both datasets must cover the same station set (both filtered to USABLE flags)
obs_ids = set(obs['station_id'])
sg_ids  = set(sg['station_id'])
only_obs = obs_ids - sg_ids
only_sg  = sg_ids - obs_ids

if only_obs:
    log.warning(f'{len(only_obs)} stations in observed but not in SoilGrids — will have NaN for sg_* cols')
if only_sg:
    log.warning(f'{len(only_sg)} stations in SoilGrids but not in observed — excluded from unified output')

print(f'Observed (USABLE)  : {len(obs):,} stations')
print(f'SoilGrids          : {len(sg):,} stations')
print(f'Only in observed   : {len(only_obs)}')
print(f'Only in SoilGrids  : {len(only_sg)}')

2026-03-19 23:18:43,251 [INFO] Observed stations loaded: 48,435 (after QC filter)


2026-03-19 23:18:43,268 [INFO] SoilGrids stations loaded: 48,435


Observed (USABLE)  : 48,435 stations
SoilGrids          : 48,435 stations
Only in observed   : 0
Only in SoilGrids  : 0


In [3]:
# ── Merge observed + SoilGrids on station_id ──────────────────────────────
# Left join: observed is the anchor. SoilGrids fills in where available.
# Stations in observed but missing from SoilGrids get NaN for all sg_* columns.

meta_cols = ['station_id', 'lat', 'lon', 'iso3', 'country', 'QC_FLAG']

# SoilGrids columns to merge (exclude metadata already in obs)
sg_value_cols = [c for c in sg.columns if c.startswith('sg_')]

merged = obs.merge(
    sg[['station_id'] + sg_value_cols],
    on='station_id',
    how='left'
)

log.info(f'Merged table: {len(merged):,} stations × {len(merged.columns)} columns')
assert len(merged) == len(obs), 'STOP: Merge changed row count — investigate duplicates in SoilGrids table'

print(f'Merged: {len(merged):,} stations × {len(merged.columns)} columns')

2026-03-19 23:18:43,660 [INFO] Merged table: 48,435 stations × 43 columns


Merged: 48,435 stations × 43 columns


In [4]:
# ── Apply fallback: observed priority, SoilGrids fill ─────────────────────
# For each property in SG_FALLBACK_MAP:
#   sg_scaled = sg_value × scale_factor  (unit conversion where needed)
#   unified_value = observed if observed is not NaN, else sg_scaled
# Source flag columns: {prop}_src = 'observed' | 'soilgrids' | 'missing'

unified = merged[meta_cols].copy()

source_records = []  # for long-format flag table

# --- Properties with SoilGrids fallback ---
for wosis_prop, (sg_col, scale_factor) in SG_FALLBACK_MAP.items():
    obs_vals = merged[wosis_prop].copy()
    sg_vals  = merged[sg_col].copy() * scale_factor  # apply unit conversion

    # Unified value: observed first, SoilGrids where observed is NaN
    unified_vals = obs_vals.where(obs_vals.notna(), other=sg_vals)

    # Source flag
    src = pd.Series('missing', index=merged.index)
    src = src.where(unified_vals.isna(), 'soilgrids')   # default: soilgrids (if non-NaN)
    src = src.where(obs_vals.isna(), 'observed')         # override: observed if obs was present

    unified[wosis_prop]          = unified_vals
    unified[f'{wosis_prop}_src'] = src

    # Append to source records (long format)
    for flag in ('observed', 'soilgrids', 'missing'):
        n = (src == flag).sum()
        source_records.append({'property': wosis_prop, 'source': flag, 'n_stations': n})

    n_obs = (src == 'observed').sum()
    n_sg  = (src == 'soilgrids').sum()
    n_mis = (src == 'missing').sum()
    scale_note = f'  [×{scale_factor}]' if scale_factor != 1.0 else ''
    log.info(f'  {wosis_prop:20s}: obs={n_obs:5,}  sg={n_sg:5,}  missing={n_mis:5,}{scale_note}')

# --- Properties with NO SoilGrids fallback (observed only) ---
for prop in OBS_ONLY_PROPS:
    if prop not in merged.columns:
        log.warning(f'  {prop}: not found in observed table — skipped')
        continue
    obs_vals = merged[prop].copy()
    src = obs_vals.apply(lambda v: 'observed' if pd.notna(v) else 'missing')
    unified[prop]          = obs_vals
    unified[f'{prop}_src'] = src

    n_obs = obs_vals.notna().sum()
    n_mis = obs_vals.isna().sum()
    log.info(f'  {prop:20s}: obs={n_obs:5,}  sg=    0  missing={n_mis:5,}  [obs-only]')

# --- SoilGrids-only extra columns (no fallback role, just preserved) ---
# sg_occ (ocd layer, g/dm³) and sg_WR_volumetric2 (0.33 bar, cm³/cm³ — NOT scaled)
for raw_col, final_col in SG_ONLY_COLS.items():
    if raw_col in merged.columns:
        unified[final_col] = merged[raw_col]
    elif final_col in merged.columns:
        unified[final_col] = merged[final_col]

log.info(f'Integration complete: {len(unified):,} stations × {len(unified.columns)} columns')
print(f'\nUnified table: {len(unified):,} stations × {len(unified.columns)} columns')

2026-03-19 23:18:43,802 [INFO]   BD                  : obs=  445  sg=46,751  missing=1,239


2026-03-19 23:18:43,828 [INFO]   CEC                 : obs=11,999  sg=35,420  missing=1,016


2026-03-19 23:18:43,853 [INFO]   CF                  : obs=9,166  sg=38,191  missing=1,078


2026-03-19 23:18:43,877 [INFO]   clay                : obs=26,561  sg=21,208  missing=  666


2026-03-19 23:18:43,901 [INFO]   N                   : obs=15,694  sg=31,767  missing=  974


...

2026-03-19 23:18:44,236 [INFO]   nematode            : obs=  686  sg=    0  missing=47,749  [obs-only]


2026-03-19 23:18:44,253 [INFO]   P                   : obs=9,710  sg=    0  missing=38,725  [obs-only]


2026-03-19 23:18:44,271 [INFO]   TC                  : obs=2,939  sg=    0  missing=45,496  [obs-only]


2026-03-19 23:18:44,273 [INFO] Integration complete: 48,435 stations × 56 columns



Unified table: 48,435 stations × 56 columns


In [5]:
# ── Build long-format source flag table ───────────────────────────────────
# Columns: station_id, property, source ('observed' | 'soilgrids' | 'missing')
# One row per station × property.

all_props = list(SG_FALLBACK_MAP.keys()) + OBS_ONLY_PROPS
src_cols  = [f'{p}_src' for p in all_props if f'{p}_src' in unified.columns]

flags_wide = unified[['station_id'] + src_cols].copy()
flags_wide.columns = ['station_id'] + all_props[:len(src_cols)]

flags_long = flags_wide.melt(
    id_vars='station_id',
    var_name='property',
    value_name='source'
)

log.info(f'Source flags table: {len(flags_long):,} rows')
print(f'Source flags (long): {len(flags_long):,} rows')
print(flags_long['source'].value_counts().to_string())

2026-03-19 23:18:44,416 [INFO] Source flags table: 1,162,440 rows


Source flags (long): 1,162,440 rows
source
missing      591599
soilgrids    348760
observed     222081


In [6]:
# ── Summary statistics ─────────────────────────────────────────────────────
n_stations = len(unified)
value_cols = [c for c in unified.columns if not c.endswith('_src') and c not in
              ['station_id', 'lat', 'lon', 'iso3', 'country', 'QC_FLAG']]

print('=== NB05 SUMMARY ===')
print(f'Total stations    : {n_stations:,}')
print(f'Countries         : {unified["iso3"].nunique()}')
print(f'Property columns  : {len(value_cols)}')
print()
print(f'{"Property":20s}  {"Obs":>7}  {"SoilGrids":>9}  {"Missing":>8}  {"Coverage%":>9}  Scale')
print('-' * 72)

for prop in list(SG_FALLBACK_MAP.keys()) + OBS_ONLY_PROPS:
    src_col = f'{prop}_src'
    if src_col not in unified.columns:
        continue
    src = unified[src_col]
    n_obs = (src == 'observed').sum()
    n_sg  = (src == 'soilgrids').sum()
    n_mis = (src == 'missing').sum()
    pct   = 100 * (n_obs + n_sg) / n_stations
    if prop in SG_FALLBACK_MAP:
        _, scale = SG_FALLBACK_MAP[prop]
        scale_str = f'×{scale:.0f}' if scale != 1.0 else '×1'
        fallback_marker = ' ←SG'
    else:
        scale_str = '—'
        fallback_marker = '    '
    print(f'{prop:20s}  {n_obs:7,}  {n_sg:9,}  {n_mis:8,}  {pct:8.1f}%{fallback_marker}  {scale_str}')

print()
print('Coverage gain from SoilGrids fallback:')
for prop, (sg_col, scale) in SG_FALLBACK_MAP.items():
    src_col = f'{prop}_src'
    n_sg    = (unified[src_col] == 'soilgrids').sum()
    n_obs   = (unified[src_col] == 'observed').sum()
    pct_obs = 100 * n_obs / n_stations
    pct_uni = 100 * (n_obs + n_sg) / n_stations
    gain    = pct_uni - pct_obs
    print(f'  {prop:20s}: {pct_obs:5.1f}% obs → {pct_uni:5.1f}% unified  (+{gain:.1f}%)')

log.info('NB05 summary computed')

=== NB05 SUMMARY ===
Total stations    : 48,435
Countries         : 23
Property columns  : 26

Property                  Obs  SoilGrids   Missing  Coverage%  Scale
------------------------------------------------------------------------
BD                        445     46,751     1,239      97.4% ←SG  ×1
CEC                    11,999     35,420     1,016      97.9% ←SG  ×1
CF                      9,166     38,191     1,078      97.8% ←SG  ×1
clay                   26,561     21,208       666      98.6% ←SG  ×1
N                      15,694     31,767       974      98.0% ←SG  ×1
occ                    29,402     18,414       619      98.7% ←SG  ×1
pH                     39,314      8,739       382      99.2% ←SG  ×1
sand                   10,252     37,162     1,021      97.9% ←SG  ×1
silt                   25,939     21,809       687      98.6% ←SG  ×1
WR_volumetric             429     46,772     1,234      97.5% ←SG  ×100
WR_gravimetric          4,765     42,527     1,143      97.6%

  CEC                 :  24.8% obs →  97.9% unified  (+73.1%)


2026-03-19 23:18:44,739 [INFO] NB05 summary computed


  CF                  :  18.9% obs →  97.8% unified  (+78.9%)
  clay                :  54.8% obs →  98.6% unified  (+43.8%)
  N                   :  32.4% obs →  98.0% unified  (+65.6%)
  occ                 :  60.7% obs →  98.7% unified  (+38.0%)
  pH                  :  81.2% obs →  99.2% unified  (+18.0%)
  sand                :  21.2% obs →  97.9% unified  (+76.7%)
  silt                :  53.6% obs →  98.6% unified  (+45.0%)
  WR_volumetric       :   0.9% obs →  97.5% unified  (+96.6%)
  WR_gravimetric      :   9.8% obs →  97.6% unified  (+87.8%)


In [7]:
# ── Save outputs ──────────────────────────────────────────────────────────
out_unified = INTER_DIR / 'unified_surface_all.parquet'
out_flags   = INTER_DIR / 'unified_source_flags.parquet'

unified.to_parquet(out_unified, index=False)
flags_long.to_parquet(out_flags, index=False)

log.info(f'Saved unified_surface_all.parquet    — {len(unified):,} rows, {out_unified.stat().st_size / 1e6:.2f} MB')
log.info(f'Saved unified_source_flags.parquet   — {len(flags_long):,} rows, {out_flags.stat().st_size / 1e6:.2f} MB')
log.info('NB05 completed successfully — proceed to NB06')

print(f'Saved: unified_surface_all.parquet    {out_unified.stat().st_size / 1e6:.2f} MB')
print(f'Saved: unified_source_flags.parquet   {out_flags.stat().st_size / 1e6:.2f} MB')

2026-03-19 23:18:45,276 [INFO] Saved unified_surface_all.parquet    — 48,435 rows, 2.70 MB


2026-03-19 23:18:45,277 [INFO] Saved unified_source_flags.parquet   — 1,162,440 rows, 2.89 MB


2026-03-19 23:18:45,278 [INFO] NB05 completed successfully — proceed to NB06


Saved: unified_surface_all.parquet    2.70 MB
Saved: unified_source_flags.parquet   2.89 MB
